In [ ]:
import sqlite3 # Para manejar la base de datos SQLite

In [2]:
database = sqlite3.connect('ejemplo.db')  # Conectar a la base de datos (o crearla si no existe)
cursor = database.cursor()  # Crear un cursor para ejecutar comandos SQL. Un cursor es un objeto que permite interactuar con la base de datos.

In [3]:
# Crear una tabla llamada 'productos' si existe que la borre y la vuelva a crear
drop_table_query = "DROP TABLE IF EXISTS productos"
cursor.execute(drop_table_query)
create_table_query = '''CREATE TABLE productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL
)'''
cursor.execute(create_table_query)
database.commit()  # Guardar los cambios en la base de datos

In [4]:
# Introducir algunos datos de ejemplo
productos = [
    ('Manzana', 0.5),
    ('Banana', 0.3),
    ('Naranja', 0.7)
]

cursor.execute("INSERT INTO productos (nombre, precio) VALUES (?, ?)", ('Silla', 45.0))
cursor.executemany("INSERT INTO productos (nombre, precio) VALUES (?, ?)", productos)
print(cursor.lastrowid)  # Imprimir el ID de la última fila insertada (4)
database.commit()  # Guardar los cambios en la base de datos

1


In [6]:
# Obtener y mostrar los datos
lista_productos = cursor.execute("SELECT * FROM productos").fetchall()
print(lista_productos)
producto_3 = cursor.execute("SELECT * FROM productos WHERE id = ?", (3,)).fetchone()
print(producto_3)

[(1, 'Silla', 45.0), (2, 'Manzana', 0.5), (3, 'Banana', 0.3), (4, 'Naranja', 0.7)]
(3, 'Banana', 0.3)


In [7]:
# Obtener todos los productos y sumar sus precios
total_precio = sum([fila[2] for fila in cursor.execute("SELECT * FROM productos")])
for fila in lista_productos:
    print(fila)
print("Precio total:", total_precio)

(1, 'Silla', 45.0)
(2, 'Manzana', 0.5)
(3, 'Banana', 0.3)
(4, 'Naranja', 0.7)
Precio total: 46.5


In [9]:
sumar_dos_ultimos = sum([fila[2] for fila in lista_productos[-2:]])
print("Suma de los precios de los dos últimos productos:", sumar_dos_ultimos)

Suma de los precios de los dos últimos productos: 1.0


In [11]:
# Actualizar el precio de un producto
cursor.execute("UPDATE productos SET precio = ?, nombre=? WHERE id = ?", (1.0, 'Plátano', 3))  # Cambiar el precio de la banana a 1.0 y el nombre a 'Plátano'
database.commit()  # Guardar los cambios en la base de datos

In [13]:
# Borrar un producto
cursor.execute("DELETE FROM productos WHERE id = ?", (4,))
database.commit()

In [16]:
productos_comienzan_s = cursor.execute("select * from productos where nombre like 's%'").fetchall()
print(len(productos_comienzan_s))
print(productos_comienzan_s)

1
[(1, 'Silla', 45.0)]


In [17]:
cursor.close()  # Cerrar el cursor
database.close()  # Cerrar la conexión a la base de datos

### Uso de dataclasses para almacenar los datos de las Selects

In [19]:
import sqlite3 # Para manejar la base de datos SQLite
from dataclasses import dataclass

@dataclass
class Producto:
    id: int
    nombre: str
    precio: float
def product_factory(cursor, row):
    return Producto(id=row[0], nombre=row[1], precio=row[2])

# Conectar a la base de datos (o crearla si no existe)
database = sqlite3.connect('ejemplo.db')  
database.row_factory = product_factory  # Configurar la fábrica de filas para devolver objetos Producto
cursor = database.cursor()  # Crear un cursor para ejecutar comandos SQL. Un cursor es un objeto que permite interactuar con la base de datos.
productos = cursor.execute("SELECT * FROM productos").fetchall()

productos.sort(key=lambda p: p.precio, reverse=True) # Ordenar productos por precio descendente
for producto in productos:
    print(f"ID: {producto.id}, Nombre: {producto.nombre}, Precio: {producto.precio}")

ID: 1, Nombre: Silla, Precio: 45.0
ID: 3, Nombre: Plátano, Precio: 1.0
ID: 2, Nombre: Manzana, Precio: 0.5
